# w9_fullcorpus_build.ipynb — 全库(23k)测试 · 阶段1 数据

User (2026-07-23):突破 2020,做全库测试。**完全不过滤**(min_length=0,
min_count=1)→ 真·全库 ~23,107 游戏。原始文本已在桶
`build_new_gamedata/`(Mendeley 目录 + games.json + kaggle prepared),
直接 stage,不重下 Mendeley。

策略:**端到端重跑 build.py 产出单一 23k embedding_h5**(避免 pool-merge
的 offset-rebase 风险),full_pool 从它直接出。代价=重嵌 2020 的 73M 句,
换来干净单一 source-of-truth。

**先出规模**(cell 4 metadata):量化"完全不过滤"的嵌入成本(游戏/评论/
句数/稀疏游戏比例)——看了规模再跑 GPU 重的 split/embed。稀疏游戏(总句
数≤2·W=32)将在阶段2走纯-CE fallback(无强增强视图)。AUTO-STOPS 关闭。


In [ ]:
# constants
import os
REPO = "/workspace/stable-query-latent"
URL  = "https://github.com/Nice9Tian/stable-query-latent.git"
BUCKET = "s3://0wov6gbp6j"
ENDPOINT = "https://s3api-us-ks-2.runpod.io"
RAW_PREFIX = f"{BUCKET}/stable-query-latent/game_review_data/build_new_gamedata"
DATA_DIR = "/workspace/fullcorpus_data"     # build.py --data-dir (network volume)
DATA_SRC = "/workspace/fusion_cache_w9"     # where full_pool + assets live
MIN_LENGTH, MIN_COUNT = 0, 1                # NO FILTER (user decree)
VIEW_W = 16                                 # sparse threshold = 2*W = 32 sentences
os.makedirs(DATA_DIR, exist_ok=True)
print(f"full-corpus build -> {DATA_DIR}, no filter (min_length={MIN_LENGTH}, "
      f"min_count={MIN_COUNT}); sparse if total sentences <= {2*VIEW_W}")

In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git fetch origin main && git reset --hard origin/main && git rev-parse --short HEAD
for pkg in ("h5py", "pandas", "wtpsplit"):
    if importlib.util.find_spec(pkg) is None:
        print(f"[warn] {pkg} missing -- build.py stages will pip-install as needed")
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)
print("repo synced")

In [ ]:
# Stage the raw review text from the bucket (Mendeley dir + games.json +
# kaggle prepared). build.py auto-reuses these (source1_done / kaggle reuse),
# so no Mendeley re-download and no Kaggle credentials are needed.
import subprocess, os
from pathlib import Path
os.makedirs(DATA_DIR, exist_ok=True)
MEND = "Steam Games Metadata and Player Reviews (2020\u20132024"   # en-dash, no ')'
targets = [
    (f"{RAW_PREFIX}/{MEND}/", f"{DATA_DIR}/{MEND}/"),
    (f"{RAW_PREFIX}/kaggle_steam_reviews_prepared/",
     f"{DATA_DIR}/kaggle_steam_reviews_prepared/"),
]
for src, dst in targets:
    print(f"sync {src} ...", flush=True)
    subprocess.run(["aws", "s3", "sync", src, dst, "--endpoint-url", ENDPOINT,
                    "--only-show-errors"], check=True)
subprocess.run(["aws", "s3", "cp", f"{RAW_PREFIX}/games.json",
                f"{DATA_DIR}/games.json", "--endpoint-url", ENDPOINT,
                "--only-show-errors"], check=True)
gr = Path(DATA_DIR) / MEND / "Game Reviews"
n_csv = len(list(gr.glob("*.csv"))) if gr.exists() else 0
print(f"staged; Mendeley Game Reviews CSVs: {n_csv}")

In [ ]:
# STAGE metadata (NO FILTER) + full-corpus SCALE report. This is the cost
# tell for 'no filter': how many games/reviews/sentences the 23k corpus has
# and what fraction are sparse (<=32 sentences -> plain-CE fallback arm).
import subprocess, sys, json, glob, os
from pathlib import Path
cmd = [sys.executable, "game_review_data/build.py", "--data-dir", DATA_DIR,
       "--only", "metadata", "--min-length", str(MIN_LENGTH),
       "--min-count", str(MIN_COUNT), "--metadata-workers", "16"]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

# scale report from the cleaned metadata JSONs (cleaned_3 = [descs..., reviews...])
meta_dir = Path(DATA_DIR) / "metadata"
files = sorted(glob.glob(str(meta_dir / "*.json")))
print(f"\n=== FULL-CORPUS SCALE (no filter) ===\nkept games: {len(files)}")
import numpy as np
rc = []                                   # review count per game (post-cleaned)
for f in files:
    d = json.loads(Path(f).read_text())
    revs = d if isinstance(d, list) else d.get("reviews", [])
    # cleaned_3 prepends up to 3 description strings; reviews are the rest
    rc.append(max(0, len(revs) - 3))
rc = np.array(rc)
print(f"reviews (post-clean): {int(rc.sum()):,}  median/game {int(np.median(rc))}")
for thr in (1, 2, 5, 10, 50, 500):
    print(f"  games with >= {thr:4d} reviews: {int((rc>=thr).sum()):,}")
print("NOTE: sentence counts (and the <=32 sparse share) are known only after "
      "the split stage; run cell 6 for that. Embedding cost scales with total "
      "sentences.")

In [ ]:
# SPLIT (SaT sentence segmentation) over the full corpus. Big -- chunk to
# avoid the half-precision OOM on huge files (hard-won lesson: --chunk-budget
# 2000 + per-chunk cache clear). GPU. Long.
import subprocess, sys
cmd = [sys.executable, "game_review_data/build.py", "--data-dir", DATA_DIR,
       "--only", "split", "--chunk-budget", "2000"]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
# sentence-scale + sparse share now computable
import glob, json
from pathlib import Path
import numpy as np
sd = Path(DATA_DIR) / "sentences"
sc = []
for f in sorted(glob.glob(str(sd / "*.json"))):
    d = json.loads(Path(f).read_text())
    sents = d if isinstance(d, list) else d.get("sentences", d.get("reviews", []))
    sc.append(len(sents))
sc = np.array(sc)
print(f"\n=== SENTENCE SCALE ===\ntotal sentences: {int(sc.sum()):,}")
print(f"sparse games (<= {2*VIEW_W} sentences -> plain-CE): "
      f"{int((sc<=2*VIEW_W).sum()):,} / {len(sc):,} "
      f"({(sc<=2*VIEW_W).mean():.1%})")

In [ ]:
# TEXT-H5 + EMBED-H5 over the full corpus (Qwen3-Embedding-0.6B, 1024-d fp16,
# on-pod GPU). Longest stage; cost scales with total sentences from cell 6.
import subprocess, sys
cmd = [sys.executable, "game_review_data/build.py", "--data-dir", DATA_DIR,
       "--only", "text-h5", "embed-h5", "--backend", "local",
       "--embedding-dtype", "float16",
       "--embedding-h5", f"{DATA_DIR}/embedding_h5.h5"]
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)
import h5py
with h5py.File(f"{DATA_DIR}/embedding_h5.h5", "r") as h:
    print(f"embedding_h5: games {h['game_names'].shape[0]:,}  "
          f"sentences {h['vectors'].shape[0]:,}  dim {h['vectors'].shape[1]}")

In [ ]:
# FULL_POOL build @23k (embedding_h5 -> flat fp16 npy + meta), streamed into
# DATA_SRC, then upload pool + h5 to the bucket for the training pods. Mirrors
# w9_a100 cell-5; NG is whatever the 23k h5 holds -- no code change for scale.
import os, time, subprocess
from pathlib import Path
import numpy as np, h5py
H5 = f"{DATA_DIR}/embedding_h5.h5"
dst_v = Path(DATA_SRC) / "full_pool_fp16.npy"
dst_m = Path(DATA_SRC) / "full_pool_meta.npz"
ready = Path(DATA_SRC) / "full_pool_READY"
os.makedirs(DATA_SRC, exist_ok=True)
with h5py.File(H5, "r") as h:
    N = h["vectors"].shape[0]
    np.savez(dst_m,
             game_review_offsets=h["game_review_offsets"][:],
             review_offsets=h["review_offsets"][:],
             game_names=np.array([g.decode() if isinstance(g, bytes) else str(g)
                                  for g in h["game_names"][:]], dtype=object))
    tmp_v = dst_v.with_suffix(".npy.tmp")
    out = np.lib.format.open_memmap(tmp_v, mode="w+", dtype=np.float16,
                                    shape=(N, 1024))
    t0 = time.time(); CH = 1 << 20
    for i in range(0, N, CH):
        out[i:i+CH] = h["vectors"][i:i+CH]
        if i % (16 << 20) == 0:
            print(f"  {i:,}/{N:,} ({(time.time()-t0)/60:.1f} min)", flush=True)
    out.flush(); del out
    os.replace(tmp_v, dst_v)
ready.write_text(f"fullcorpus {N}")
print(f"full_pool @ {N:,} sentences built -> {dst_v}")
# upload pool + h5 so the (separate) training pods can stage them
for f in (dst_v, dst_m):
    subprocess.run(["aws", "s3", "cp", str(f),
                    f"{BUCKET}/fusion_cache_w9/{f.name}",
                    "--endpoint-url", ENDPOINT, "--only-show-errors"], check=True)
subprocess.run(["aws", "s3", "cp", H5,
                f"{BUCKET}/fullcorpus/embedding_h5.h5",
                "--endpoint-url", ENDPOINT, "--only-show-errors"], check=True)
print("uploaded full_pool + embedding_h5 to bucket")

In [ ]:
# AUTO-STOP the pod (artifacts are on the network volume + bucket).
import subprocess
print("stage-1 data build complete; stopping pod")
try:
    subprocess.run(["runpodctl", "stop", "pod",
                    __import__("os").environ.get("RUNPOD_POD_ID", "")], check=False)
except Exception as e:
    print("auto-stop skipped:", e)